# သင်ခန်းစာ ၁၁ - မော်ဒယ် ဆက်သွယ်မှု စံနှုန်း (MCP)

**မော်ဒယ် ဆက်သွယ်မှု စံနှုန်း (MCP)** သည် အေးဂျင့်များအနေဖြင့် အလုပ်ဆောင်ချိန်တွင် ကိရိယာများ၊ အရင်းအမြစ်များနှင့် ဒေတာရင်းမြစ်များကို လှုပ်ရှားမှုအလျောက် ရှာဖွေနိုင်စေသော ဖွင့်လှစ်စံနှုန်း ဖြစ်သည်။ အေးဂျင့်အတွင်းတွင် ကိရိယာများကို သေချာပေးသတ်မှတ်ခြင်း မလုပ်ဘဲ MCP သည် လိုအပ်သည့်အခါ အင်တာနက်ဆာဗာများနှင့် ချိတ်ဆက်နိုင်စေသည်။

ဤသင်ခန်းစာတွင် သင်လေ့လာရမည့်အချက်များမှာ-
- MCP ဆိုတာဘာလဲ၊ အေးဂျင့်စနစ်များအတွက် ဘာကြောင့် အရေးကြီးသနည်း
- MCP client-server ဖွဲ့စည်းပုံဘယ်လို အလုပ်လုပ်သလဲ
- MCP ပုံစံဖြင့် ကိရိယာရှာဖွေမှု အေးဂျင့်များကို မည်သို့ တည်ဆောက်ကြမလဲ


## ပြင်ဆင်ခြင်း

**လိုအပ်သော အခြေခံများ:**
- မိုဒယ်တစ်ခု တပ်ဆင်ပြီး Microsoft Foundry ပရောဂျက်
- အတည်ပြုဖို့ `az login` ကို အသုံးပြုပါ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Model Context Protocol (MCP) ဆိုတာဘာလဲ?

MCP က AI ကိုယ်စားလှယ်တွေကို ပြင်ပကိရိယာတွေအဖြစ်သော်လည်း အချက်အလက်ရင်းမြစ်တွေကို ရှာဖွေပြီး အပြန်အလှန် ဆက်သွယ်နိုင်ဖို့ စံချိန်စံညွှန်းတစ်ခုသတ်မှတ်ပေးတယ်။

- **MCP ဆာဗာ**: ကိရိယာတွေ၊ အရင်းအမြစ်တွေနဲ့ prompt တွေကို စံ protocol တစ်ခုဖြင့် ထုတ်ပြန်ပေးတယ်
- **MCP Client**: ဆာဗာတွေနဲ့ချိတ်ဆက်ပြီး ဗဟိုမှတ်တမ်းတွင် ရနိုင်သော အင်အားများကို ရှာဖွေ သုံးနိုင်သော ကိုယ်စားလှယ် runtime ဖြစ်တယ်
- **Dynamic Discovery**: ကိုယ်စားလှယ်တွေဟာ အတိအကျစီစဉ်ထားတဲ့ ကိရိယာတွေ မလိုအပ်ဘဲ အချိန်မှာရရှိနိုင်တာတွေကို ရှာဖွေတတ်တယ်

ဒီဟာက ကိုယ်စားလှယ်ကုဒ် မပြောင်းလဲဘဲ အသစ်ထည့်သွင်းနိုင်တဲ့ဖွံ့ဖြိုးဆဲ ကိုယ်စားလှယ် စနစ်များ ဖန်တီးရာမှာ အလွန်အစွမ်းထက်တယ်။


## MCP သည် မည်သို့ စွမ်းဆောင်သလဲ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. အေးဂျင့် (MCP client) သည် MCP ဆာဗာနှင့် ချိတ်ဆက်သည်
2. ဆာဗာသည် ရရှိနိုင်သည့်ကိရိယာများနှင့် ၎င်းတို့၏ schema များစာရင်းဖြင့် ပြန်ကြားသည်
3. အေးဂျင့်သည် ၎င်း၏ ချဉ်းကပ်နည်းအတွင်း တွေ့ရှိနိုင်သည့် ကိရိယာများကို ခေါ်ယူနိုင်သည်
4. ရလဒ်များသည် အတူတူ protocol ဖြင့် ပြန်လည်ထွက်ပေါက်သွားသည်


## MCP ကိရိယာရှာဖွေမှုကို ကူးမယ်

အမှန်တကယ် MCP ဆာဗာက ဆာဗာလုပ်ငန်းဖြစ်နေတာလို မရှိမဖြစ်လိုအပ်လို၍ MCP ဆက်သွယ်ထားတဲ့ အဆင့်အတန်း सेवाကို လုပ်ဆောင်မယ့် `@tool` လုပ်ဆောင်ချက်တွေဖြင့် ဒီပုံစံကို ပြသမှာ ဖြစ်တယ်။

ထုတ်လုပ်မှုမှာတော့ ဒီကိရိယာတွေကို ဒေသတွင်းမှာပြတ်မဲ့မလုပ်ကြဘဲ MCP ဆာဗာထံမှ အလိုအလျောက် ရှာဖွေနိုင်မှာ ဖြစ်ပါတယ်။


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-စတိုင်ကိရိယာများဖြင့် Agent တစ်ဦးတည်ဆောက်ခြင်း


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ထုတ်လုပ်မှုတွင် MCP

ထုတ်လုပ်မှုပတ်ဝန်းကျင်တွင် MCP သည် စွမ်းအားဖြင့် ယုတ်မာသော ပုံစံများကို ခွင့်ပြုသည်-

- **Dynamic tool discovery**: အေးဂျင့်များသည် MCP ဆာဗာများနှင့်ချိတ်ဆက်ကာ ကိရိယာများကို runtime တွင် ရှာဖွေသည်
- **Decoupled architecture**: ကိရိယာပံ့ပိုးသူများသည် အေးဂျင့်၏ ဆက်စပ်မဖြစ်ပေါ်စေရန် အဆင့်တက် ကန့်သတ်ထားသည်
- **Cross-organization sharing**: အဖွဲ့များသည် MCP ဆာဗာများမှတဆင့် မည်သည့်အေးဂျင့်မှမဆို အသုံးပြုနိုင်သော စွမ်းဆောင်ရည်များကို ဖော်ပြနိုင်သည်
- **Microsoft Agent Framework support**: MAF တွင် `mcp` ပေါင်းစပ်မှုဖြင့် MCP client အထောက်အပံ့ ထည့်သွင်းထားသည်

MAF တို့နှင့် အသုံးပြုရန် MCP ဆာဗာတစ်ခုကို အသုံးပြုသည့်အခါ `hosted_mcp_tool()` သို့မဟုတ် MCP client ပေါင်းစပ်မှုဖြင့် ချိတ်ဆက်နိုင်သည်။

**ပိုမိုသိရှိလိုပါက-**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## အကျဉ်းချုပ်

ယခုသင်ခန်းစာတွင် သင်တန်းသားတစ်ဦးအနေဖြင့် လေ့လာသင်ကြားရသော အကြောင်းအရာများမှာ
- **MCP** သည် အေးဂျင့်များနှင့် ကိရိယာပေးသူများအကြား dynamic tool တွေရှာဖွေရေးအတွက် ဖွင့်လှစ်သော စံတော်ချိန်တစ်ခုဖြစ်သည်။
- **client-server architecture** က အေးဂျင့်များကို runtime အချိန်တွင် အရည်အချင်းများရှာဖွေရန်ခွင့်ပြုသည်။
- MCP သည် code ပြင်ဆင်မှုမလိုပါဘဲ ကိရိယာများကို ထည့်သွင်းနိုင်သော **ချိတ်ဆက်ခြင်းမတည့်သော၊ ကျယ်ပြန့်စွာတိုးချဲ့နိုင်သော အေးဂျင့်စနစ်များ** ကို ခွင့်ပြုသည်။
- Microsoft Agent Framework သည် ထုတ်လုပ်မှုအတွက် **ထည့်သွင်းထားသော MCP ပံ့ပိုးမှု** ကို ပံ့ပိုးပေးသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
